# USD/INR basis study

Daily basis curve, DNS estimates and risk checks.

## 1. Imports and settings

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import data_prep as dp
import curve_build as cb
import curve_model as cm
import curve_estimate as ce

pd.set_option("display.max_columns", 30)

In [ ]:
PROJECT = Path.cwd()
RAW = PROJECT / "raw"

CURVE_FILE = PROJECT / "basis_curve_long.csv"
PARAM_FILE = PROJECT / "params.csv"
MONTHLY_FILE = PROJECT / "panel_monthly.csv"

DATA_MODE = "csv"       # "csv" or "raw"
PARAM_MODE = "saved"    # "saved" or "fit"

START = "2018-07-30"
END = None
TRAIN_END = "2023-12-31"
TEST_START = "2024-01-01"

TENORS = [f"{m}M" for m in range(1, 13)]
LAM = 0.30
INR_BENCHMARK = "ois"   # "ois" or "tbill"

MAXITER = 1000
TWO_ROUTES = True
RUN_COMPARISONS = False
TURN_WALK = True
USE_FBD = True
RUN_RISK = False
RUN_STRESS = False
RUN_LOCAL_PROJECTIONS = False
FBD_FALLBACK = False
RUN_SIMULATION = False
RUN_TURN_TEST = False
RUN_NULL_SIMULATIONS = False
SIM_REPS = 10
SEED = 1000

RISK_TENORS = ["1M", "3M", "12M"]
RISK_LEVELS = [0.95, 0.99]
RISK_WINDOW = 250
RISK_MODELS = ["RW-normal", "HS", "EWMA-normal", "EWMA-t-cal",
               "SS-normal", "SS-t-EWMA", "SS-FHS"]
RISK_RM_LAMBDA = 0.94
RISK_EWMA_GRID = [0.60, 0.70, 0.80, 0.85, 0.90, 0.94, 0.97]

STRESS_HORIZON = 10
NOTIONAL_USD = 10_000_000
EPISODES = {
    "Mar-Apr 2020": ("2020-03-01", "2020-04-30"),
    "Dec 2025": ("2025-12-01", "2025-12-31"),
    "Mar-Apr 2026": ("2026-03-16", "2026-04-30")
}

LP_HORIZONS = range(7)
LP_SE = "HC1"
SAVE_RESULTS = False

Use saved basis data and parameter estimates for quick runs. Select `raw` or `fit` in settings only when needed.

## 2. Basis data

In [ ]:
if DATA_MODE == "csv":
    if not CURVE_FILE.exists():
        raise FileNotFoundError(f"Basis file not found: {CURVE_FILE}")
    curve = pd.read_csv(CURVE_FILE, parse_dates=["date"])
    diagnostics = None
    rate_checks = None

elif DATA_MODE == "raw":
    FORWARD_FILE = RAW / "4e05a313-FORWARD_PREMIA.csv"
    OIS_FILE = RAW / "fbil_mibor_ois.csv"
    TBILL_FILE = RAW / "9890848d-tbillindia.csv"
    PAR_FILE = RAW / "us_treasury_par_yield_1990_2023.csv"

    FRED_FILES = {
        "1M": RAW / "ef3dafc8-DGS1MO.csv",
        "3M": RAW / "bfa0fe33-DGS3MO.csv",
        "6M": RAW / "58f3b4e6-DGS6MO.csv",
        "12M": RAW / "43066850-DGS1.csv"
    }

    OIS_DAYS = {
        "1M": 365 / 12, "2M": 365 / 6,
        "3M": 365 / 4, "6M": 365 / 2,
        "9M": 365 * 0.75, "1Y": 365
    }

    TBILL_DAYS = {"7 Days": 7, "14 Days": 14}
    for month in range(1, 13):
        word = "Month" if month == 1 else "Months"
        TBILL_DAYS[f"{month} {word}"] = 365 * month / 12

    forward_data = dp.load_fbil_forwards(FORWARD_FILE)
    ois = dp.load_long_curve(
        OIS_FILE, OIS_DAYS, tenor_col="Period", skiprows=2
    )
    tbill = dp.load_long_curve(
        TBILL_FILE, TBILL_DAYS, tenor_col="Tenor", skiprows=2
    )
    us = dp.load_us_treasury(PAR_FILE, FRED_FILES)

    BILL_FILES = [RAW / f"us_treasury_bill_rates_{year}.csv"
                  for year in (2024, 2025, 2026)]
    bills = cb.load_us_bills(BILL_FILES) if all(p.exists() for p in BILL_FILES) else None

    inr = ois if INR_BENCHMARK == "ois" else tbill
    curve, diagnostics = cb.build(
        forwards=forward_data,
        inr_curve=inr,
        usd_curve=us,
        ois_curve=ois,
        tbill_curve=tbill,
        start=START,
        end=END,
        tenors=TENORS,
        include_fbd=True,
        fallback_april1=FBD_FALLBACK
    )
    rate_checks = cb.rate_checks(curve, us, bills)

else:
    raise ValueError("DATA_MODE must be 'csv' or 'raw'")

curve = curve[curve["date"] >= pd.Timestamp(START)]
if END is not None:
    curve = curve[curve["date"] <= pd.Timestamp(END)]

curve = curve.sort_values(["date", "n"]).reset_index(drop=True)
print("Rows:", len(curve))
print("Dates:", curve["date"].nunique())
print("Missing basis:", curve["basis_bp"].isna().sum())
display(curve.head())

In [ ]:
plot_tenors = [t for t in ["1M", "3M", "6M", "12M"] if t in TENORS]
plot_data = curve.pivot(
    index="date", columns="tenor", values="basis_bp"
)

ax = plot_data[plot_tenors].plot(figsize=(11, 4), lw=0.8)
ax.set_title("USD/INR forward basis")
ax.set_ylabel("Annualized basis points")
plt.show()

## 3. Model inputs

In [ ]:
data = cm.load_curve(curve, TENORS)
train = cm.sample(data, start=START, end=TRAIN_END)

print("Full sample:", data["basis"].index.min().date(),
      "to", data["basis"].index.max().date())
print("Training sample:", train["basis"].index.min().date(),
      "to", train["basis"].index.max().date())
print("Basis matrix:", data["basis"].shape)
print("Parameters:", len(cm.parameter_names(TENORS)))

display(data["basis"].head())

## 4. Parameters

In [ ]:
names = cm.parameter_names(TENORS)

if PARAM_MODE == "saved":
    if not PARAM_FILE.exists():
        raise FileNotFoundError(f"Parameter file not found: {PARAM_FILE}")

    table = pd.read_csv(PARAM_FILE, index_col=0)
    if TURN_WALK and USE_FBD:
        param_column = "main"
    elif not TURN_WALK and USE_FBD:
        param_column = "one_charge_per_season"
    elif TURN_WALK and not USE_FBD:
        param_column = "curve_only"
    else:
        raise ValueError("Combined variant needs fresh estimation")

    if param_column not in table.columns:
        raise ValueError(f"Missing saved parameters: {param_column}")
    params = table.loc[names, param_column].to_numpy(dtype=float)
    fitted_train = cm.run_filter(params, train, lam=LAM, use_fbd=USE_FBD)

    summary = pd.DataFrame([{
        "fit": f"{param_column} (saved)",
        "loglik": fitted_train.llf,
        "converged": np.nan
    }])
    fits = None

elif PARAM_MODE == "fit":
    fits, table, summary = ce.estimate(
        train,
        lam=LAM,
        maxiter=MAXITER,
        two_routes=TWO_ROUTES,
        comparisons=RUN_COMPARISONS,
        turn_walk=TURN_WALK,
        use_fbd=USE_FBD
    )
    params = table.loc[names, "main"].to_numpy(dtype=float)

else:
    raise ValueError("PARAM_MODE must be 'saved' or 'fit'")

display(summary)
display(pd.Series(params, index=names, name="estimate").to_frame())

## 5. Factors and forecast errors

In [ ]:
curve_only_params = None
if USE_FBD and TURN_WALK and "curve_only" in table.columns:
    curve_only_params = table.loc[names, "curve_only"].to_numpy(dtype=float)

out = ce.analyse(
    data,
    params,
    curve_only_params=curve_only_params,
    lam=LAM,
    use_fbd=USE_FBD
)

print("Filtered factors")
display(out["filtered"].tail())

print("Common-factor fit by tenor")
display(out["fit_by_tenor"])

print("Forecast-error correlations")
display(out["innovation_corr"])
display(pd.Series(out["innovation_white"], name="value"))

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(11, 9), sharex=True)

for ax, name in zip(axes, cm.STATES):
    ax.plot(out["filtered"].index, out["filtered"][name], lw=0.9)
    ax.set_title(name.capitalize())
    ax.axhline(0, color="grey", lw=0.5)

axes[-1].set_xlabel("Date")
fig.tight_layout()
plt.show()

In [ ]:
print("Turn estimates by financial year")
display(out["turn_by_year"])

print("FBD-based turn comparison")
display(out["turn_validation"])

## 6. Basis drivers

In [ ]:
import curve_drivers as cd

allocation = cd.covariance_allocation(data)
persistence = cd.half_lives(pd.Series(params, index=names))

display(allocation.query("period == 'full'"))
display(persistence)

In [ ]:
lp = None

if RUN_LOCAL_PROJECTIONS:
    if not MONTHLY_FILE.exists():
        raise FileNotFoundError(f"Monthly data not found: {MONTHLY_FILE}")

    monthly = pd.read_csv(
        MONTHLY_FILE, index_col="date", parse_dates=True
    )
    drivers = cd.prepare_drivers(monthly)

    lp = cd.projections(
        smoothed=out["smoothed"],
        drivers=drivers,
        horizons=LP_HORIZONS,
        se_type=LP_SE
    )
    display(lp)

## 7. Risk backtest

In [ ]:
risk = None

if RUN_RISK:
    import risk_backtest as rb

    years = out["turn_by_year"]
    turn_avg = years.loc[
        years["fiscal_year"] <= pd.Timestamp(TRAIN_END).year,
        "two_step"
    ].mean()

    if pd.isna(turn_avg):
        raise ValueError("No training-period turn estimates")

    risk = rb.run(
        one_step=out["one_step"],
        turn_avg=turn_avg,
        reset_days=out["reset_days"],
        tenors=RISK_TENORS,
        train_end=TRAIN_END,
        test_start=TEST_START,
        levels=RISK_LEVELS,
        window=RISK_WINDOW,
        models=RISK_MODELS,
        rm_lambda=RISK_RM_LAMBDA,
        ewma_grid=RISK_EWMA_GRID
    )

    display(risk["backtests"])
    display(risk["loss_comparison"])

## 8. Turn and stress

In [ ]:
stress_report = None

if RUN_STRESS:
    import stress_turn as st

    stress_report = st.run(
        data=data,
        filtered=out["filtered"],
        smoothed=out["smoothed"],
        observed=out["observed_turn"],
        rollover_days=out["rollover_days"],
        episodes=EPISODES,
        horizon=STRESS_HORIZON,
        lam=LAM,
        notional_usd=NOTIONAL_USD
    )

    display(stress_report["turn_now"])
    display(stress_report["event_summary"])
    display(stress_report["stress_scenarios"])

## 9. Simulation and turn test

Simulation refits the model for each replication. Keep this off during normal runs. Ten replications are only useful for checking that the code runs.

In [ ]:
simulation = None
null_simulations = None

if RUN_SIMULATION or RUN_NULL_SIMULATIONS:
    if not TURN_WALK or not USE_FBD:
        raise ValueError("Simulation section uses the full model with FBD")
    import curve_simulation as cs

    if RUN_SIMULATION:
        simulation = cs.run(
            data=train,
            estimated=params,
            scenario="baseline",
            reps=SIM_REPS,
            seed=SEED,
            lam=LAM,
            maxiter=MAXITER,
            two_routes=TWO_ROUTES
        )
        display(simulation["params"])

    if RUN_NULL_SIMULATIONS:
        null_simulations = cs.run(
            data=train,
            estimated=params,
            scenario="no_turn",
            reps=SIM_REPS,
            seed=SEED,
            lam=LAM,
            maxiter=MAXITER,
            two_routes=TWO_ROUTES
        )
        display(null_simulations["params"])

In [ ]:
turn_test = None

if RUN_TURN_TEST:
    if not TURN_WALK or not USE_FBD:
        raise ValueError("Turn LR test requires the full turn model with FBD")
    import curve_turn_test as ct

    train_result = cm.run_filter(params, train, lam=LAM)

    turn_test = ct.run(
        data=train,
        full_params=params,
        full_loglik=train_result.llf,
        null_simulations=null_simulations,
        seed=SEED,
        lam=LAM,
        maxiter=MAXITER,
        full_converged=(
            bool(fits["main"].mle_retvals.get("converged", False))
            if fits is not None else None
        )
    )
    display(turn_test)
    print("Monte Carlo p-value:", turn_test.loc[0, "mc_p_value"])
    print("Null replications:", turn_test.loc[0, "mc_reps"])

## 10. Save results

In [ ]:
run_folder = None

if SAVE_RESULTS:
    stamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S_%f")
    run_folder = PROJECT / "results" / f"basis_{stamp}"
    run_folder.mkdir(parents=True, exist_ok=False)

    curve.to_csv(run_folder / "basis_curve_long.csv", index=False)
    table.to_csv(run_folder / "params.csv")
    summary.to_csv(run_folder / "fit_summary.csv", index=False)
    out["filtered"].to_csv(run_folder / "states_filtered.csv")
    out["smoothed"].to_csv(run_folder / "states_smoothed.csv")
    out["one_step"].to_csv(run_folder / "one_step_forecasts.csv", index=False)
    out["turn_by_year"].to_csv(run_folder / "turn_by_year.csv", index=False)
    allocation.to_csv(run_folder / "covariance_allocation.csv", index=False)
    persistence.to_csv(run_folder / "factor_half_lives.csv", index=False)

    if lp is not None:
        lp.to_csv(run_folder / "local_projections.csv", index=False)
    if risk is not None:
        risk["backtests"].to_csv(run_folder / "risk_backtest.csv", index=False)
        risk["loss_comparison"].to_csv(run_folder / "risk_loss_comparison.csv", index=False)
        risk["tuning"].to_csv(run_folder / "risk_tuning.csv", index=False)
    if stress_report is not None:
        stress_report["stress_scenarios"].to_csv(
            run_folder / "stress_scenarios.csv", index=False
        )
    if turn_test is not None:
        turn_test.to_csv(run_folder / "turn_lr_test.csv", index=False)
    if null_simulations is not None:
        null_simulations["params"].to_csv(
            run_folder / "null_simulation_params.csv", index=False
        )
    if rate_checks is not None:
        rate_checks.to_csv(run_folder / "rate_checks.csv", index=False)
    if simulation is not None:
        simulation["params"].to_csv(
            run_folder / "simulation_params.csv", index=False
        )

    settings = {
        "data_mode": DATA_MODE,
        "param_mode": PARAM_MODE,
        "basis_file": str(CURVE_FILE),
        "parameter_file": str(PARAM_FILE),
        "start": START,
        "end": END,
        "train_end": TRAIN_END,
        "test_start": TEST_START,
        "tenors": TENORS,
        "lambda": LAM,
        "inr_benchmark": INR_BENCHMARK,
        "fbd_fallback": FBD_FALLBACK,
        "maxiter": MAXITER,
        "two_routes": TWO_ROUTES,
        "comparisons": RUN_COMPARISONS,
        "run_risk": RUN_RISK,
        "run_stress": RUN_STRESS,
        "run_projections": RUN_LOCAL_PROJECTIONS,
        "run_simulation": RUN_SIMULATION,
        "run_turn_test": RUN_TURN_TEST,
        "run_null_simulations": RUN_NULL_SIMULATIONS,
        "sim_reps": SIM_REPS,
        "seed": SEED,
        "risk_tenors": RISK_TENORS,
        "risk_levels": RISK_LEVELS,
        "risk_window": RISK_WINDOW,
        "risk_models": RISK_MODELS,
        "risk_rm_lambda": RISK_RM_LAMBDA,
        "risk_ewma_grid": RISK_EWMA_GRID,
        "turn_walk": TURN_WALK,
        "use_fbd": USE_FBD,
        "stress_horizon": STRESS_HORIZON,
        "stress_episodes": EPISODES,
        "notional_usd": NOTIONAL_USD,
        "lp_horizons": list(LP_HORIZONS),
        "lp_se": LP_SE
    }

    with (run_folder / "settings.json").open("x", encoding="utf-8") as f:
        json.dump(settings, f, indent=2)

    print("Saved to", run_folder)